# Cloud Architecture & Deployment — Applied

**AI Architecture · Week 20b**

Offline notebook for deploying the Week 19b insurance-underwriter RAG assistant on Azure: topology, Bicep-shaped IaC, cost estimation, release orchestration, GitHub Actions workflow, and FDE hand-off checklist.

## 1. Deployment topology overview

```mermaid
flowchart TB
  U[Underwriter] --> FD[Azure Front Door WAF]
  FD --> CA[Azure Container Apps FastAPI RAG]
  CA --> PG[(Postgres Flexible Server + pgvector)]
  CA --> AOAI[Azure OpenAI GPT-4o]
  CA --> BLOB[(Blob audit + prompt registry)]
  CA --> KV[Key Vault]
  CA --> AI[Application Insights]
  CA -. blocked .-> NET[Public internet]
```

Production replaces public service calls with Private Endpoints, private DNS, managed identity, and egress lockdown.

## 2. Bicep-shaped IaC example with commentary

In [ ]:
bicep_example = '''
param location string = resourceGroup().location
param environment string
param containerImage string
param promptRegistryBlobUrl string
param vectorIndexName string
param chatModelDeployment string = 'gpt-4o-prod'
param embeddingModelDeployment string = 'text-embedding-3-large-prod'

resource kv 'Microsoft.KeyVault/vaults@2023-07-01' = {
  name: 'kv-ins-rag-${environment}'
  location: location
  properties: { tenantId: subscription().tenantId sku: { family: 'A' name: 'standard' } enableRbacAuthorization: true }
}

resource pg 'Microsoft.DBforPostgreSQL/flexibleServers@2023-06-01-preview' = {
  name: 'pg-ins-rag-${environment}'
  location: location
  sku: { name: 'Standard_D4s_v3' tier: 'GeneralPurpose' }
  properties: { version: '16' storage: { storageSizeGB: 128 } publicNetworkAccess: 'Disabled' }
}

resource app 'Microsoft.App/containerApps@2024-03-01' = {
  name: 'ca-ins-rag-${environment}'
  location: location
  identity: { type: 'SystemAssigned' }
  properties: {
    configuration: {
      activeRevisionsMode: 'Multiple'
      ingress: { external: false targetPort: 8000 }
      registries: [{ server: 'acrinsrag.azurecr.io' identity: 'system' }]
    }
    template: {
      containers: [{
        name: 'rag-api'
        image: containerImage
        env: [
          { name: 'PROMPT_REGISTRY_BLOB_URL' value: promptRegistryBlobUrl }
          { name: 'VECTOR_INDEX_NAME' value: vectorIndexName }
          { name: 'CHAT_MODEL_DEPLOYMENT' value: chatModelDeployment }
          { name: 'EMBEDDING_MODEL_DEPLOYMENT' value: embeddingModelDeployment }
        ]
      }]
      scale: { minReplicas: 0 maxReplicas: 10 rules: [{ name: 'http' http: { metadata: { concurrentRequests: '25' } } }] }
    }
  }
}

// Private endpoint modules would connect app subnet private DNS zones to Key Vault, Postgres, Blob, ACR, and Azure OpenAI.
'''
print(bicep_example)
for phrase in ["publicNetworkAccess: 'Disabled'", 'SystemAssigned', 'activeRevisionsMode', 'PROMPT_REGISTRY_BLOB_URL']:
    print('contains', phrase, phrase in bicep_example)

## 3. Deployment topology + cost estimator snippet

In [ ]:
from __future__ import annotations
from pydantic import BaseModel, Field

PRICES = {'gpt4o_input_per_mtok':2.50,'gpt4o_output_per_mtok':10.00,'embedding_3_large_per_mtok':0.13,'containerapp_replica_month':75.0,'postgres_gp_2vc_month':320.0,'postgres_gp_4vc_month':620.0,'postgres_gp_8vc_month':1240.0,'blob_hot_gb_month':0.018,'frontdoor_month':165.0,'log_analytics_ingest_gb':2.76,'private_endpoint_month':7.30,'ptu_month':6000.0}
class Service(BaseModel):
    name: str; sku: str; replicas: int = 1; private_endpoint: bool = True
class Workload(BaseModel):
    label: str; users: int; questions_per_user_day: int = 40; business_days_per_month: int = 22; prompt_tokens: int = 1500; response_tokens: int = 800; cache_hit_rate: float = Field(0.0, ge=0.0, le=0.95); use_ptu: bool = False; log_gb_per_day: float = 1.0
class DeploymentTopology(BaseModel):
    name: str; region: str; services: list[Service]; docs: int = 40_000; chunks_per_doc: int = 10; embedding_dims: int = 1536; avg_chunk_tokens: int = 800; blob_gb: float = 100.0; private_networking: bool = True; public_egress_allowed: bool = False
    prompt_registry_blob_url: str; vector_index_name: str; chat_model_deployment: str; embedding_model_deployment: str
    def estimate_monthly_cost(self, w: Workload):
        q = w.users*w.questions_per_user_day*w.business_days_per_month; billable=q*(1-w.cache_hit_rate)
        input_mtok=billable*w.prompt_tokens/1_000_000; output_mtok=billable*w.response_tokens/1_000_000
        chunks=self.docs*self.chunks_per_doc; embed_mtok=chunks*self.avg_chunk_tokens/1_000_000; vector_gb=chunks*self.embedding_dims*4/1_000_000_000
        pg_key='postgres_gp_8vc_month' if w.users>=500 else 'postgres_gp_4vc_month' if w.users>=200 else 'postgres_gp_2vc_month'
        replicas=10 if w.users>=500 else 5 if w.users>=200 else 2
        model=PRICES['ptu_month'] if w.use_ptu else input_mtok*PRICES['gpt4o_input_per_mtok']+output_mtok*PRICES['gpt4o_output_per_mtok']
        lines={'Azure OpenAI generation or PTU':model,'One-time corpus embedding amortized':embed_mtok*PRICES['embedding_3_large_per_mtok']/12,'Container Apps compute':replicas*PRICES['containerapp_replica_month'],'PostgreSQL Flexible Server pgvector':PRICES[pg_key],'Blob Storage audit and prompt registry':self.blob_gb*PRICES['blob_hot_gb_month'],'Azure Front Door Premium estimate':PRICES['frontdoor_month'],'Log Analytics ingest':w.log_gb_per_day*30*PRICES['log_analytics_ingest_gb'],'Private Endpoints':sum(s.private_endpoint for s in self.services)*PRICES['private_endpoint_month']}
        return {'workload':w.label,'queries_month':int(q),'billable_queries_after_cache':int(billable),'raw_vector_gb':round(vector_gb,2),'line_items':{k:round(v,2) for k,v in lines.items()},'total':round(sum(lines.values()),2),'network_isolation_ok':self.private_networking and not self.public_egress_allowed}

topology = DeploymentTopology(name='insurance-underwriter-rag', region='eastus2', services=[Service(name=n, sku=s) for n,s in [('rag-api','Container Apps'),('aoai','Azure OpenAI'),('postgres','Flexible Server'),('audit-blob','StorageV2'),('key-vault','standard'),('app-insights','workspace'),('acr','Premium')]], prompt_registry_blob_url='https://stpromptprod.blob.core.windows.net/prompts/registry.json', vector_index_name='underwriting-index-2026-07-17', chat_model_deployment='gpt-4o-prod', embedding_model_deployment='text-embedding-3-large-prod')
for scenario in [Workload(label='Baseline', users=50, cache_hit_rate=.10, log_gb_per_day=1.5), Workload(label='Growth', users=200, cache_hit_rate=.20, use_ptu=True, log_gb_per_day=4), Workload(label='Enterprise', users=500, cache_hit_rate=.25, use_ptu=True, log_gb_per_day=8)]:
    r=topology.estimate_monthly_cost(scenario)
    print()
    print(f"{r['workload']} total=${r['total']:,.2f} queries={r['queries_month']:,} vectors={r['raw_vector_gb']}GB isolation={r['network_isolation_ok']}")
    for k,v in r['line_items'].items(): print(f"  {k:42s} ${v:,.2f}")

## 4. Release pipeline orchestrator with three scenarios

In [ ]:
from dataclasses import dataclass
from typing import Literal
Axis = Literal['prompt','model','index']; Decision = Literal['PROMOTE','HOLD','ROLLBACK']
@dataclass(frozen=True)
class VersionTuple: prompt: str; model: str; index: str
@dataclass(frozen=True)
class Metrics: latency_p95_ms: int; error_rate: float; groundedness_score: float; cost_per_query: float
@dataclass(frozen=True)
class Guardrails: max_latency_p95_ms: int=8500; max_error_rate: float=.02; min_groundedness_score: float=.86; max_cost_per_query: float=.08
@dataclass(frozen=True)
class StageResult: traffic_percent: int; decision: Decision; reasons: list[str]
def judge(m, g):
    reasons=[]
    if m.latency_p95_ms>g.max_latency_p95_ms: reasons.append(f'latency {m.latency_p95_ms}>{g.max_latency_p95_ms}')
    if m.error_rate>g.max_error_rate: reasons.append(f'error_rate {m.error_rate:.3f}>{g.max_error_rate:.3f}')
    if m.groundedness_score<g.min_groundedness_score: reasons.append(f'groundedness {m.groundedness_score:.2f}<{g.min_groundedness_score:.2f}')
    if m.cost_per_query>g.max_cost_per_query: reasons.append(f'cost {m.cost_per_query:.3f}>{g.max_cost_per_query:.3f}')
    return ('ROLLBACK' if reasons else 'PROMOTE'), reasons or ['all guardrails passed']
def orchestrate(current, changes, metrics, g=Guardrails()):
    proposed=VersionTuple(changes.get('prompt',current.prompt), changes.get('model',current.model), changes.get('index',current.index))
    axes=[a for a in ('prompt','model','index') if getattr(current,a)!=getattr(proposed,a)]
    stages=[]; rollback=set()
    for pct,m in zip((10,50,100), metrics):
        decision,reasons=judge(m,g); stages.append(StageResult(pct,decision,reasons))
        if decision=='ROLLBACK':
            if any('groundedness' in r for r in reasons) and 'prompt' in axes: rollback.add('prompt')
            elif any(('latency' in r or 'cost' in r) for r in reasons) and 'model' in axes: rollback.add('model')
            else: rollback.update(axes)
            break
    plan={a:(f'rollback {a} to {getattr(current,a)}; preserve other healthy axes' if a in rollback else f'keep {getattr(proposed,a)}') for a in axes}
    return proposed, axes, stages, plan
current=VersionTuple('prompt-v17','gpt-4o-prod','index-2026-07-17')
scenarios=[('healthy release', {'prompt':'prompt-v18'}, [Metrics(6200,.006,.91,.031),Metrics(6500,.007,.90,.032),Metrics(6800,.008,.90,.033)]),('prompt groundedness regression', {'prompt':'prompt-v18'}, [Metrics(6100,.006,.80,.031)]),('model latency regression', {'model':'gpt-4o-2024-11-prod'}, [Metrics(9400,.006,.90,.041)])]
for name,changes,metrics in scenarios:
    proposed, axes, stages, plan=orchestrate(current, changes, metrics)
    print()
    print(name, 'axes=', axes, 'proposed=', proposed)
    for s in stages: print(f'  {s.traffic_percent}% {s.decision} {s.reasons}')
    print('  rollback plan:', plan)

## 5. GitHub Actions workflow YAML

In [ ]:
workflow = '''
name: deploy-insurance-rag
on:
  pull_request:
  push:
    branches: [main]
  workflow_dispatch:
    inputs:
      promote_prod:
        type: boolean
        default: false
jobs:
  pr-quality:
    if: github.event_name == 'pull_request'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.11' }
      - run: pip install -r requirements-dev.txt
      - run: pytest -q
      - run: ruff check .
      - run: python evals/run_regression.py --golden evals/golden_underwriting.json --min-groundedness 0.86
  build-staging:
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    environment: staging
    steps:
      - uses: actions/checkout@v4
      - uses: azure/login@v2
        with: { client-id: '${{ secrets.AZURE_CLIENT_ID }}', tenant-id: '${{ secrets.AZURE_TENANT_ID }}', subscription-id: '${{ secrets.AZURE_SUBSCRIPTION_ID }}' }
      - run: az acr build -r acrinsrag -t rag-api:${{ github.sha }} .
      - run: az acr repository show-manifests -n acrinsrag --repository rag-api
      - run: az containerapp update -g rg-ins-rag-staging -n ca-ins-rag-staging --image acrinsrag.azurecr.io/rag-api:${{ github.sha }} --revision-suffix ${{ github.sha }}
      - run: python tests/online_smoke.py --env staging
  promote-prod:
    needs: build-staging
    if: github.event_name == 'workflow_dispatch' && inputs.promote_prod == true
    runs-on: ubuntu-latest
    environment: production
    steps:
      - uses: actions/checkout@v4
      - uses: azure/login@v2
        with: { client-id: '${{ secrets.AZURE_CLIENT_ID }}', tenant-id: '${{ secrets.AZURE_TENANT_ID }}', subscription-id: '${{ secrets.AZURE_SUBSCRIPTION_ID }}' }
      - run: python deploy/set_traffic.py --app ca-ins-rag-prod --revision ${{ github.sha }} --traffic 10
      - run: python deploy/watch_canary.py --max-p95-ms 8500 --min-groundedness 0.86 --max-error-rate 0.02
      - run: python deploy/set_traffic.py --app ca-ins-rag-prod --revision ${{ github.sha }} --traffic 50
      - run: python deploy/watch_canary.py --max-p95-ms 8500 --min-groundedness 0.86 --max-error-rate 0.02
      - run: python deploy/set_traffic.py --app ca-ins-rag-prod --revision ${{ github.sha }} --traffic 100
      - run: python deploy/rollback_on_metric_breach.py --axes prompt,model,index
'''
print(workflow)
print('has eval gate:', 'evals/run_regression.py' in workflow)
print('has staged canary:', '--traffic 10' in workflow and '--traffic 50' in workflow and '--traffic 100' in workflow)

## 6. FDE hand-off checklist

In [ ]:
checklist = [
    'C4 deployment diagram with Private Endpoints and private DNS zones',
    'Bicep modules plus dev/staging/prod parameter files',
    'Managed identity RBAC matrix for Key Vault, Blob, ACR, Postgres, and monitoring',
    'Data-boundary table showing tokens, PII, prompts, retrieved chunks, and audit records',
    'Budget model for Standard vs PTU, Container Apps, Postgres, Front Door, Blob, and Log Analytics',
    'GitHub Actions gates for pytest, ruff, golden evals, image scan, staging smoke, and prod canary',
    'Prompt/model/index release registry and rollback runbook',
    '2am drill evidence: rollback target, owner, metric, and expected recovery under three minutes',
]
for i, item in enumerate(checklist, 1):
    print(f'{i}. {item}')

## Exercises

1. Add `dev`, `staging`, and `prod` parameter values for `minReplicas`, `maxReplicas`, and `logSamplingRate`.
2. Change Growth from PTU to Standard and compare cost versus latency-risk notes.
3. Add a fourth release scenario where a shadow index loses recall and only the index alias rolls back.
4. Write the boundary table for exactly what crosses Azure OpenAI and what remains only in Blob audit.

## Links
- Literature note: `02 Literature Notes/AI Architecture/Cloud Architecture & Deployment — Applied`
- Snippets: `04 Code Snippets/AI Architecture/AI Week 20b Azure Deployment Topology Cost Estimator`, `.../AI Week 20b Prompt Model Index Release Orchestrator`
- MOC: `06 Maps of Content/AI Architecture Concepts`